# Steam games' metadata

## 1. Retrieving the list of apps from Steam

Steam provides a list of all available apps and their _appid_,
which includes games but also dlc, demos, music etc.
We cannot obtain just the list of games,
so we have to query each app to check its type.

In [1]:
import json
import requests
import time

# Json file containing the list of Steam apps (appids + names)
applist_file = "applist.json"

# Json file documenting the queries made to the Steam API
file_queries = "./queries.json"

# Directory storing retrieve games' metadata
metadata_dir = "./raw_metadata_dataset/"


/Users/jessiegalasso-carbonnel/PyCharmMiscProject/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Retrieving json file with the list of all _appids_:

In [2]:
try:
    with open(applist_file, "x") as f:
        print("> Request to steam api")
        x = requests.get('http://api.steampowered.com/ISteamApps/GetAppList/v2/?format=json')
        applist = json.loads(x.text)
        f.write(json.dumps(applist))
except Exception:
    print("> File already retrieved, loading content")
    with open(applist_file) as f:
        applist = json.loads(f.read())

print("> Number of entries in the list of apps: "+ str(len(applist["applist"]["apps"])))
print("\nShort extract:")
print(applist["applist"]["apps"][0:15])


> File already retrieved, loading content
> Number of entries in the list of apps: 257148

Short extract:
[{'appid': 5, 'name': 'Dedicated Server'}, {'appid': 7, 'name': 'Steam Client'}, {'appid': 8, 'name': 'winui2'}, {'appid': 10, 'name': 'Counter-Strike'}, {'appid': 20, 'name': 'Team Fortress Classic'}, {'appid': 30, 'name': 'Day of Defeat'}, {'appid': 40, 'name': 'Deathmatch Classic'}, {'appid': 50, 'name': 'Half-Life: Opposing Force'}, {'appid': 60, 'name': 'Ricochet'}, {'appid': 70, 'name': 'Half-Life'}, {'appid': 80, 'name': 'Counter-Strike: Condition Zero'}, {'appid': 90, 'name': 'Half-Life Dedicated Server'}, {'appid': 92, 'name': 'Codename Gordon'}, {'appid': 100, 'name': 'Counter-Strike: Condition Zero Deleted Scenes'}, {'appid': 130, 'name': 'Half-Life: Blue Shift'}]


Buiding the list of _appids_:

In [3]:
appid_list = [item['appid'] for item in applist["applist"]["apps"]]
print(len(appid_list))
print(appid_list[0:15])

257148
[5, 7, 8, 10, 20, 30, 40, 50, 60, 70, 80, 90, 92, 100, 130]


Testing that there are no duplicates in the list of appids

In [4]:
if(len(appid_list) != len(set(appid_list))):
    print("[ERROR] found some duplicates in the retrieved list of appids")
else:
    print("[OK] No duplicates found in the list of appids")

[OK] No duplicates found in the list of appids


Create or retrieve the json file listing the appids for which we already made a query

In [5]:
try:
    with open(file_queries, "x") as f:
        print("> Create json file storing info regarding the queries made to retrieve metadata")
        queries = {}
        f.write(json.dumps(queries))
except Exception:
    print("> Loading list of queries made to retrieve metadata")
    with open(file_queries) as f:
        queries = json.loads(f.read())

print("Sent queries to retrieve game metadata: " +str(len(queries)))

proc_appids = list(queries.keys())


def print_progress_bar(percentage, bar_length=30):
    filled_length = int(bar_length * percentage // 100)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    print(f'Progress: |{bar}| {percentage:.1f}%')
print_progress_bar((len(queries) * 100) / len(appid_list))  # 65% progress


> Loading list of queries made to retrieve metadata
Sent queries to retrieve game metadata: 256007
Progress: |█████████████████████████████-| 99.6%


## Loop to retrieve the metadata

In [6]:
def query_appid_metadata(appid):
    query = {}

    try:
        url = f"https://store.steampowered.com/api/appdetails?appids={appid}"
        print(url)
        response = requests.get(url)
        while response.status_code == 429:
            print("[WARNING] API rate limit exceeded, sleep for 60 seconds")
            time.sleep(60)
            response = requests.get(url)

        data = response.json()
    except:
        print("[ERROR] Failed to process query for appid " + str(appid))
        return False, None, None

    success = False
    metadata = None
    if str(appid) in data:
        if data[str(appid)]['success']:
            success = True
            metadata = data[str(appid)]['data']
            query["type"] = metadata['type']
    else:
        query["info"] = ("Queried appid different from the one in the query result")

    query["success"] = success
    return True, query, metadata


def save_game_metadata(appid, metadata):
    file_name = str(appid) + "__" + metadata["name"] + ".json"
    try:
        with open(metadata_dir + file_name, "x") as f:
            f.write(json.dumps(metadata, indent=4))
    except Exception:
        print("[Error] File already exists: " + file_name)

def update_queries(queries):
    with open(file_queries, "w") as f:
        f.write((json.dumps(queries)))

In [7]:
try:
    for appid in appid_list:
        if str(appid) not in proc_appids:
            success, query, metadata = query_appid_metadata(appid)
            if success:
                if metadata is not None and query["type"] == "game":
                    save_game_metadata(appid, metadata)
                queries[appid] = query
                proc_appids.append(appid)
            else:
                break
except Exception as e:
        print("[Error] "+ e)

update_queries(queries)

print("Travail terminé!")

https://store.steampowered.com/api/appdetails?appids=3845180
https://store.steampowered.com/api/appdetails?appids=3845280
https://store.steampowered.com/api/appdetails?appids=3845310
https://store.steampowered.com/api/appdetails?appids=3845320
https://store.steampowered.com/api/appdetails?appids=3845330
https://store.steampowered.com/api/appdetails?appids=3845390
https://store.steampowered.com/api/appdetails?appids=3845400
https://store.steampowered.com/api/appdetails?appids=3845420
https://store.steampowered.com/api/appdetails?appids=3845430
https://store.steampowered.com/api/appdetails?appids=3845450
https://store.steampowered.com/api/appdetails?appids=3845460
https://store.steampowered.com/api/appdetails?appids=3845480
https://store.steampowered.com/api/appdetails?appids=3845490
https://store.steampowered.com/api/appdetails?appids=3845520
https://store.steampowered.com/api/appdetails?appids=3845540
https://store.steampowered.com/api/appdetails?appids=3845570
https://store.steampower